# Variant Effect Analysis with Genova

This notebook demonstrates:

1. Loading a synthetic VCF file
2. Predicting variant pathogenicity
3. Generating SHAP-style explanations
4. Visualising variant effects

All data is synthetic -- no real genome files needed.

## 1. Setup

In [ ]:
import numpy as np
import torch

from genova.utils.config import GenovaConfig
from genova.data.tokenizer import GenomicTokenizer
from genova.models.model_factory import create_model
from genova.api.inference import InferenceEngine

# Create a small model
config = GenovaConfig()
config.model.d_model = 128
config.model.n_heads = 4
config.model.n_layers = 2
config.model.d_ff = 512

engine = InferenceEngine(device="cpu", config=config)
engine.load()
print("Engine loaded.")

## 2. Create Synthetic VCF Data

In [ ]:
# Synthetic VCF content
vcf_content = """##fileformat=VCFv4.2
##source=synthetic
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr1\t1000\t.\tA\tG\t50\tPASS\t.
chr1\t2000\t.\tC\tT\t50\tPASS\t.
chr2\t5000\t.\tG\tA\t50\tPASS\t.
chr3\t10000\t.\tT\tC\t50\tPASS\t.
chr7\t50000\t.\tAC\tA\t50\tPASS\t.
"""

# Parse VCF
variants = []
for line in vcf_content.strip().split("\n"):
    if line.startswith("#"):
        continue
    fields = line.split("\t")
    chrom, pos, _, ref, alt = fields[:5]
    variants.append({
        "chrom": chrom,
        "pos": int(pos),
        "ref": ref,
        "alt": alt,
        "key": f"{chrom}:{pos}:{ref}>{alt}",
    })

print(f"Parsed {len(variants)} variants:")
for v in variants:
    print(f"  {v['key']}")

**Expected output:**
```
Parsed 5 variants:
  chr1:1000:A>G
  chr1:2000:C>T
  chr2:5000:G>A
  chr3:10000:T>C
  chr7:50000:AC>A
```

## 3. Predict Pathogenicity

In [ ]:
# Create synthetic flanking sequences
def generate_context(ref, alt, flank_size=50):
    np.random.seed(42)
    flank = ''.join(np.random.choice(['A','C','G','T'], size=flank_size))
    ref_seq = flank + ref + flank
    alt_seq = flank + alt + flank
    return ref_seq, alt_seq

ref_seqs = []
alt_seqs = []
for v in variants:
    ref_seq, alt_seq = generate_context(v["ref"], v["alt"])
    ref_seqs.append(ref_seq)
    alt_seqs.append(alt_seq)

predictions = engine.predict_variant(ref_seqs, alt_seqs)

print("\nVariant Effect Predictions:")
print(f"{'Variant':<25} {'Score':>8} {'Label':<12} {'Confidence':>10}")
print("-" * 60)
for v, p in zip(variants, predictions):
    print(f"{v['key']:<25} {p['score']:>8.4f} {p['label']:<12} {p['confidence']:>10.4f}")

**Expected output:**
```
Variant Effect Predictions:
Variant                    Score Label        Confidence
------------------------------------------------------------
chr1:1000:A>G              0.6234 pathogenic       0.2468
chr1:2000:C>T              0.5891 pathogenic       0.1782
...
```
(Scores are from a randomly initialised model.)

## 4. SHAP-style Explanations

We approximate feature importance by masking individual positions and
measuring the change in prediction score.

In [ ]:
def compute_importance(engine, ref_seq, alt_seq, window=5):
    """Simple occlusion-based importance scores."""
    # Baseline prediction
    baseline = engine.predict_variant([ref_seq], [alt_seq])[0]["score"]

    importance = []
    seq_len = len(ref_seq)

    for i in range(0, seq_len - window + 1, window):
        # Mask region with Ns
        masked_ref = ref_seq[:i] + "N" * window + ref_seq[i+window:]
        masked_alt = alt_seq[:i] + "N" * window + alt_seq[i+window:]

        score = engine.predict_variant([masked_ref], [masked_alt])[0]["score"]
        importance.append(abs(baseline - score))

    return np.array(importance)

# Compute for first variant
imp = compute_importance(engine, ref_seqs[0], alt_seqs[0], window=10)
print(f"Importance scores ({len(imp)} windows):")
print(f"  Max importance: {imp.max():.6f} at position {imp.argmax() * 10}")
print(f"  Mean importance: {imp.mean():.6f}")

## 5. Visualise Results

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scores bar chart
    ax = axes[0]
    keys = [v["key"] for v in variants]
    scores = [p["score"] for p in predictions]
    colors = ["red" if p["label"] == "pathogenic" else "green" for p in predictions]
    ax.barh(keys, scores, color=colors, alpha=0.7)
    ax.axvline(x=0.5, color="black", linestyle="--", label="Threshold")
    ax.set_xlabel("Pathogenicity Score")
    ax.set_title("Variant Effect Predictions")
    ax.legend()

    # Importance heatmap
    ax = axes[1]
    positions = np.arange(len(imp)) * 10
    ax.bar(positions, imp, width=8, color="steelblue")
    ax.set_xlabel("Position")
    ax.set_ylabel("Importance")
    ax.set_title(f"Position Importance: {variants[0]['key']}")

    plt.tight_layout()
    plt.show()
    print("Plots generated successfully.")

except ImportError:
    print("matplotlib not installed -- skipping plots.")

## Summary

In this notebook you learned how to:
- Parse synthetic VCF data
- Predict variant pathogenicity using the Genova model
- Compute occlusion-based importance scores (SHAP-style)
- Visualise predictions and explanations

Next: See `04_generation.ipynb` for sequence generation.